## Netflix Visualization Dashboard

## Introduction
This project explores the distribution and evolution of Netflix's content catalog through interactive data visualizations. Using the Netflix Movies and Series dataset, we examine how the representation of movies, television shows, genres, content ratings, and countries differs across the catalog and changes over time.

The project combines two case studies. The first examines temporal changes in Netflix content additions, genre representation, and content-rating distributions. The second focuses on geographic representation and relationships between content characteristics.

Seven visualizations are presented within a shared dashboard, allowing users to explore the data through interactive filters. Our approach builds on previous research involving Netflix content analysis, clustering, and multidimensional interactive visualization.

Research Questions
1. How have Netflix's movie and television show additions changed over time?
2. Which genres are most frequently represented, and how have they changed?
3. How do content ratings differ between movies and television shows?
4. Which countries contribute the most content?
5. How do genre distributions differ between countries?
6. Can movie attributes reveal distinct content groups?

## Data Loading

In [30]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output, HTML
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Find the project root
project_root = Path.cwd()

if not (project_root / "data" / "netflix_cleaned.csv").exists():
    project_root = project_root.parent

# Load cleaned dataset
file_path = project_root / "data" / "netflix_cleaned.csv"

df = pd.read_csv(file_path)

# Ensure numerical columns are correctly typed
df["year_added"] = pd.to_numeric(
    df["year_added"], errors="coerce"
)

df["release_year"] = pd.to_numeric(
    df["release_year"], errors="coerce"
)

df["duration_minutes"] = pd.to_numeric(
    df["duration_minutes"], errors="coerce"
)

# Prepare categorical columns
for col in ["type", "rating", "country", "listed_in"]:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# Split multiple genres and countries
df["genre_list"] = df["listed_in"].apply(
    lambda s: [x.strip() for x in s.split(",") if x.strip()]
)

df["country_list"] = df["country"].apply(
    lambda s: [x.strip() for x in s.split(",") if x.strip()]
)

print("Dataset loaded successfully!")
print("Total titles:", len(df))
print("Movies:", (df["type"] == "Movie").sum())
print("TV Shows:", (df["type"] == "TV Show").sum())

df.head()

Dataset loaded successfully!
Total titles: 8806
Movies: 6130
TV Shows: 2676


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_number,duration_minutes,seasons,genre_list,country_list
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,90,90.0,NaN,[Documentaries],[United States]
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,2,NaN,2.0,"[International TV Shows, TV Dramas, TV Mysteries]",[South Africa]
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,1,NaN,1.0,"[Crime TV Shows, International TV Shows, TV Ac...",[Unknown]
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,1,NaN,1.0,"[Docuseries, Reality TV]",[Unknown]
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,2,NaN,2.0,"[International TV Shows, Romantic TV Shows, TV...",[India]


Smriti - Create clusters here

## Visualization 1: Movies vs. TV Shows Over Time

In [31]:
def create_chart_1(data):
    counts = (
        data.dropna(subset=["year_added"])
        .groupby(["year_added", "type"])
        .size()
        .reset_index(name="count")
    )

    fig = px.area(
        counts,
        x="year_added",
        y="count",
        color="type",
        title="1. Movies vs. TV Shows Added Over Time",
        labels={
            "year_added": "Year Added",
            "count": "Number of Titles",
            "type": "Content Type"
        }
    )

    return fig

## Visualization 2: Genre Trends Over Time

In [32]:
def create_chart_2(data):
    genres = data.explode("genre_list").copy()

    top = genres["genre_list"].value_counts().head(10).index
    genres = genres[genres["genre_list"].isin(top)]

    counts = (
        genres.groupby(["genre_list", "year_added"])
        .size()
        .unstack(fill_value=0)
    )

    counts.columns = counts.columns.astype(int)

    fig = px.imshow(
        counts,
        title="2. Genre Trends Over Time",
        labels={
            "x": "Year Added",
            "y": "Genre",
            "color": "Number of Titles"
        },
        aspect="auto",
        text_auto=True,
        color_continuous_scale="Blues"
    )

    fig.update_layout(height=550)

    return fig

## Visualization 3: Content Ratings by Type

In [33]:
def create_chart_3(data):
    counts = (
        data.groupby(["type", "rating"])
        .size()
        .reset_index(name="count")
    )

    counts["percentage"] = (
        counts["count"]
        / counts.groupby("type")["count"].transform("sum")
        * 100
    )

    fig = px.bar(
        counts,
        x="type",
        y="percentage",
        color="rating",
        title="3. Content Ratings by Type",
        labels={
            "type": "Content Type",
            "percentage": "Percentage of Titles",
            "rating": "Content Rating"
        }
    )

    fig.update_layout(
        barmode="stack",
        yaxis_range=[0, 100]
    )

    return fig

Smriti - Add Visualizations 4-6

### Visualization 7: Overall Genre Distribution

This visualization examines the most frequently represented genres in Netflix's content catalog. A horizontal bar chart is used to compare the number of titles associated with each genre, with longer bars representing higher counts. The visualization displays the twelve most common genre labels and allows users to observe how their distribution changes when dashboard filters are applied.

Since individual titles may belong to multiple genres, each title can contribute to more than one category. As a result, the combined genre counts may exceed the total number of unique titles in the selected dataset.

In [34]:
def create_chart_7(data):
    genres = data.explode("genre_list").copy()

    genres = genres.drop_duplicates(
        ["show_id", "genre_list"]
    )

    counts = (
        genres["genre_list"]
        .value_counts()
        .head(12)
        .sort_values()
        .reset_index()
    )

    counts.columns = ["genre", "count"]

    fig = px.bar(
        counts,
        x="count",
        y="genre",
        orientation="h",
        title="7. Most Common Netflix Genres",
        labels={
            "count": "Number of Titles",
            "genre": "Genre"
        },
        text="count",
        color="count",
        color_continuous_scale="Blues"
    )

    fig.update_layout(
        height=550,
        showlegend=False,
        coloraxis_showscale=False,
        xaxis_title="Number of Titles",
        yaxis_title="Genre"
    )

    fig.update_traces(
        textposition="outside",
        cliponaxis=False
    )

    return fig

In [35]:
fig = create_chart_7(df)
fig.show()

## Final Interactive Dashboard

In [36]:
# Get available years
years = sorted(
    df["year_added"].dropna().astype(int).unique()
)

min_year = int(min(years))
max_year = int(max(years))

# Collect available categories
all_genres = sorted({
    genre
    for genres in df["genre_list"]
    for genre in genres
})

all_countries = sorted({
    country
    for countries in df["country_list"]
    for country in countries
    if country != "Unknown"
})

all_ratings = sorted(df["rating"].unique())

# Year range slider
year_filter = widgets.IntRangeSlider(
    value=(max(2016, min_year), max_year),
    min=min_year,
    max=max_year,
    step=1,
    description="Years:",
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)

# Content type dropdown
type_filter = widgets.Dropdown(
    options=["All", "Movie", "TV Show"],
    value="All",
    description="Type:"
)

# Genre dropdown
genre_filter = widgets.Dropdown(
    options=["All"] + all_genres,
    value="All",
    description="Genre:"
)

# Country dropdown
country_filter = widgets.Dropdown(
    options=["All"] + all_countries,
    value="All",
    description="Country:"
)

# Rating dropdown
rating_filter = widgets.Dropdown(
    options=["All"] + all_ratings,
    value="All",
    description="Rating:"
)

In [37]:
def filter_data():
    filtered = df.copy()

    # Year range
    start_year, end_year = year_filter.value

    filtered = filtered[
        filtered["year_added"].between(
            start_year, end_year
        )
    ]

    # Content type
    if type_filter.value != "All":
        filtered = filtered[
            filtered["type"] == type_filter.value
        ]

    # Genre
    if genre_filter.value != "All":
        filtered = filtered[
            filtered["genre_list"].apply(
                lambda genres: genre_filter.value in genres
            )
        ]

    # Country
    if country_filter.value != "All":
        filtered = filtered[
            filtered["country_list"].apply(
                lambda countries:
                    country_filter.value in countries
            )
        ]

    # Content rating
    if rating_filter.value != "All":
        filtered = filtered[
            filtered["rating"] == rating_filter.value
        ]

    return filtered

Smriti - add create_chart_4 - 6 functions to be called here

In [ ]:
charts = [
    ("Figure 1", create_chart_1),
    ("Figure 2", create_chart_2),
    ("Figure 3", create_chart_3),
    ("Figure 4", create_chart_4),
    ("Figure 5", create_chart_5),
    ("Figure 6", create_chart_6),
    ("Figure 7", create_chart_7)
]

In [40]:
dashboard_output = widgets.Output()

def update_dashboard(change=None):
    filtered = filter_data()

    with dashboard_output:
        clear_output(wait=True)

        # Dashboard title
        display(HTML(
            "<h2>Netflix Catalog Analysis Dashboard</h2>"
        ))

        # Summary statistics
        total = len(filtered)
        movies = (filtered["type"] == "Movie").sum()
        shows = (filtered["type"] == "TV Show").sum()

        display(HTML(
            f"<h3>Total Titles: {total:,} | "
            f"Movies: {movies:,} | "
            f"TV Shows: {shows:,}</h3>"
        ))

        if filtered.empty:
            display(HTML(
                "<p>No titles match the selected filters.</p>"
            ))
            return

        for name, create_fig in charts:
            # Check for chart-specific missing data
            if name in ["Figure 4", "Figure 5"]:
                has_countries = filtered["country_list"].apply(
                    lambda countries: any(
                        c != "Unknown" for c in countries
                    )
                ).any()

                if not has_countries:
                    display(HTML(
                        f"<p>{name}: No country data available.</p>"
                    ))
                    continue

            if name == "Figure 6":
                has_clusters = (
                    (filtered["type"] == "Movie")
                    & (filtered["cluster"] != "Not Classified")
                ).any()

                if not has_clusters:
                    display(HTML(
                        "<p>Figure 6: No classified movies "
                        "match these filters.</p>"
                    ))
                    continue

            if name in ["Figure 2", "Figure 7"]:
                has_genres = filtered["genre_list"].apply(
                    lambda genres: len(genres) > 0
                ).any()

                if not has_genres:
                    display(HTML(
                        f"<p>{name}: No genre data available.</p>"
                    ))
                    continue

            fig = create_fig(filtered)
            display(fig)

## Interactive Dashboard Design

The final dashboard combines seven visualizations into a shared interactive environment for exploring Netflix's content catalog. Users can filter the dataset by year, content type, genre, country, and content rating. These controls update the visualizations using the same filtered dataset, allowing users to examine relationships between different content characteristics.

The visualizations include temporal comparisons, categorical distributions, geographic relationships, and movie attribute clustering. Together, they provide several perspectives on the same dataset and support the research questions developed in our initial project proposal.

In [41]:
# Register change handlers
for control in [
    year_filter,
    type_filter,
    genre_filter,
    country_filter,
    rating_filter
]:
    control.observe(update_dashboard, names="value")

# Reset button
reset_button = widgets.Button(
    description="Reset Filters",
    button_style="info"
)

def reset_filters(button):
    year_filter.value = (
        max(2016, min_year),
        max_year
    )
    type_filter.value = "All"
    genre_filter.value = "All"
    country_filter.value = "All"
    rating_filter.value = "All"

    update_dashboard()

reset_button.on_click(reset_filters)

# Organize filters
filter_layout = widgets.VBox([
    year_filter,
    widgets.HBox([type_filter, genre_filter]),
    widgets.HBox([country_filter, rating_filter]),
    reset_button
])

# Display dashboard
display(filter_layout, dashboard_output)

# Render initial charts
update_dashboard()

Output()

## Final Discussion and Conclusion

This project explored Netflix's content catalog through seven interactive visualizations, examining trends in content additions, genre representation, content ratings, geographic distribution, and movie characteristics. By combining two case studies into a single dashboard, we aimed to provide a more comprehensive understanding of how Netflix's content is distributed and how its composition has changed over time.

The first case study revealed that movies generally accounted for more annual additions than TV shows, with both categories experiencing significant growth leading up to 2019. International Movies, Dramas, and Comedies were among the most represented genres, following similar trends in annual additions. Content-rating comparisons also demonstrated that TV-MA represented a substantial portion of both movies and television shows. These observations generally aligned with the findings of Ali et al. (2025).

The second case study expanded our analysis by examining geographic representation, relationships between countries and genres, and patterns in movie characteristics through clustering. These visualizations provide additional perspectives on how Netflix's catalog differs across countries and content categories.

By combining these findings into an interactive dashboard, users can filter and compare multiple aspects of the dataset to identify relationships that may not be immediately obvious in individual charts. Although the dataset represents a historical snapshot and contains some incomplete information, the project demonstrates how data visualization can help reveal meaningful patterns in Netflix's content distribution.

## References
Netflix Movies and Series dataset sourced from Kaggle.
https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

Ali et al. (2025), Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.

## Reproducing This Project
Instructions for installing python library dependencies are located in the README of this project.